In [12]:
import pandas as pd
from deepeval.metrics import GEval
from deepeval.models import GPTModel
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from dotenv import load_dotenv
import warnings
import os

In [13]:
load_dotenv()

API_KEY = os.getenv('API_KEY')

In [14]:
warnings.filterwarnings('ignore')
os.environ["CONFIDENT_METRIC_LOGGING_VERBOSE"] = "0"

In [21]:
abstracts = pd.read_csv('/home/vinicius/Área de Trabalho/Artigos/llm_artigo/llmArtigo/datasets/Abstracts_ORA - abstracts.csv')


In [23]:
abstracts = abstracts.drop(['division', 'department'], axis=1)

In [24]:
abstracts

,title,abstract
0,Niu_2025_Functional_Nanolayer_Dielectrics,Negatively charged dielectric films have signi...
1,Brody_2025_Numerical_simulation_of,Hypersonic flight is an essential component of...
2,Ehrenfels_2025_The_epistemology_of,This thesis addresses the question ‘Under what...
3,Greenrod_2025_Temperature_as_a,Thermal change has a profound impact on specie...
4,Chao_2025_Synthetic_transmembrane_transporters,The thesis describes the design and synthesis ...
5,Gao_2025_Association_between_neuroticism,"BACKGROUND: Neuroticism, a personality trait r..."
6,Parkes_2025_Magnetically-activated_DNA_and,Nucleic acids serve as the fundamental buildin...
7,Taskesen_2025_Tuning_magnetism_and,This thesis reports the synthesis and characte...
8,Rad_2025_Police_Unionism,"In 2024, there were over 1,300 incidents of po..."
9,Zong_2025_Sustainable_removal_of,Conventional wastewater treatment often fails ...


In [25]:
df = pd.read_csv('/home/vinicius/Área de Trabalho/Artigos/llm_artigo/llmArtigo/datasets/teses1_prompt_fixo_reduzido.csv')
df.tail()

,title,gemma-3-4b-it,gemma-3-4b-it_response,gemma-3-4b-it_time,gemma-3-4b-it_n_tokens_in,gemma-3-4b-it_n_tokens_out,gemma-3-12b-it,gemma-3-12b-it_response,gemma-3-12b-it_time,gemma-3-12b-it_n_tokens_in,...,gpt-oss-120b,gpt-oss-120b_response,gpt-oss-120b_time,gpt-oss-120b_n_tokens_in,gpt-oss-120b_n_tokens_out,gpt-oss-20b,gpt-oss-20b_response,gpt-oss-20b_time,gpt-oss-20b_n_tokens_in,gpt-oss-20b_n_tokens_out
25,Liang_2025_The_effects_of,NaN,Here’s the first iteration of the PhD thesis a...,32.919435,76450.0,197.0,NaN,This research investigates the perceptual chal...,110.217091,76450.0,...,NaN,This dissertation investigates the persistent ...,40.643243,74181,1488,NaN,The present doctoral research addresses the pe...,26.538514,74181,1750
26,Wellalagodage_2025_An_analysis_of,NaN,Here's the first iteration of the PhD thesis a...,35.195420,83612.0,225.0,NaN,This thesis investigates novel approaches to n...,120.859204,83612.0,...,NaN,The accurate quantification of cardiac output ...,36.536560,80299,849,NaN,Computed cardiopulmonography (CCP) integrates ...,28.282435,80299,1632
27,Senese_2025_Structure_of_many-body,NaN,The impact of quantum entanglement on the dyna...,29.179406,77478.0,190.0,NaN,This thesis investigates the dynamical propert...,101.083898,77478.0,...,NaN,The thesis addresses the exponential complexit...,38.723544,76409,1244,NaN,Quantum many‑body dynamics governed by local H...,61.987183,76409,6382
28,Zhou_2025_Moderators_and_mediators,NaN,Here’s the first iteration of the PhD thesis a...,35.185588,100380.0,210.0,NaN,Childhood anxiety disorders are characterized ...,118.404364,100380.0,...,NaN,Child anxiety disorders affect up to one‑third...,52.345843,93857,1584,NaN,NaN,87.208315,93857,8192
29,Collingwood_2025_A_computational_model,NaN,Here's the first iteration of the PhD thesis a...,31.492604,84692.0,298.0,NaN,This research investigates the neural and comp...,101.087764,84692.0,...,NaN,"Habitual behavior, defined as stimulus‑respons...",44.262039,86271,1215,NaN,"Habits, defined as stimulus‑response (S‑R) ass...",27.846690,86271,1248


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 96 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   title                                             30 non-null     str    
 1   gemma-3-4b-it                                     0 non-null      float64
 2   gemma-3-4b-it_response                            30 non-null     str    
 3   gemma-3-4b-it_time                                30 non-null     float64
 4   gemma-3-4b-it_n_tokens_in                         30 non-null     float64
 5   gemma-3-4b-it_n_tokens_out                        30 non-null     float64
 6   gemma-3-12b-it                                    0 non-null      float64
 7   gemma-3-12b-it_response                           30 non-null     str    
 8   gemma-3-12b-it_time                               30 non-null     float64
 9   gemma-3-12b-it_n_tokens_in        

In [26]:
model = GPTModel(
    model = "gpt-5",
    temperature= 0,
    api_key= API_KEY
)

In [27]:
model_columns = [col for col in df.columns if col.endswith("_response")]
model_columns

['gemma-3-4b-it_response',
 'gemma-3-12b-it_response',
 'gemma-3-27b-it_response',
 'llama-3.3-70B-Instruct_response',
 'DeepSeek-R1-Distill-Llama-70B_response',
 'DeepSeek-R1-Distill-Qwen-32B_response',
 'mistral-small-3.2-24B-Instruct-2506_response',
 'qwen/qwen3-30b-a3b-2507_response',
 'meta-llama-3.1-8B-Instruct_response',
 'llama-3.2-1B-Instruct_response',
 'llama-3.2-3B-Instruct_response',
 'DeepSeek-R1-Distill-Llama-8B_response',
 'DeepSeek-R1-Distill-Qwen-1.5B_response',
 'DeepSeek-R1-Distill-Qwen-7B_response',
 'DeepSeek-R1-Distill-Qwen-14B_response',
 'Phi-4-mini-instruct_response',
 'qwen/qwen3-4b-2507_response',
 'gpt-oss-120b_response',
 'gpt-oss-20b_response']

In [28]:
metric = GEval(
    name="Abstract_Quality",
    model= model,
    evaluation_steps=[
        "Verify whether the generated abstract preserves the core scientific elements of the reference, including problem, objectives, methodology, results, and contributions.",
    
        "Assess semantic fidelity: check if the generated abstract accurately represents the meaning of the reference without introducing distortions or hallucinations.",
    
        "Check for missing critical information from the reference abstract and penalize significant omissions.",
    
        "Ensure the output follows the style of a scientific abstract, without bullet points, markdown formatting, reasoning traces, or meta-commentary.",
    
        "Evaluate conciseness: the abstract should be compact and avoid unnecessary verbosity or redundancy."
        ],
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT,
    ],

)

In [29]:
import pandas as pd
import os
from deepeval.test_case import LLMTestCase
from deepeval import evaluate

output_file = "geval_results_gpt.csv"

for idx in range(len(df)):
    title = df.loc[idx, "title"] if "title" in df.columns else f"thesis_{idx}"
    reference_abstract = abstracts.loc[idx, "abstract"]

    for model_col in model_columns:
        model_name = model_col.replace("_response", "")
        candidate_summary = df.loc[idx, model_col]

        # Pula inferências nulas ou vazias
        if pd.isna(candidate_summary) or str(candidate_summary).strip() == "":
            print(f"[SKIP] idx={idx} | model={model_name} → inferência vazia")
            result_row = {
                "idx":          idx,
                "title":        title,
                "model":        model_name,
                "geval_score":  None,
                "geval_reason": "skipped: empty inference"
            }
        else:
            try:
                test_case = LLMTestCase(
                    input=title,
                    actual_output=str(candidate_summary),
                    expected_output=str(reference_abstract)
                )
                metric.measure(test_case)
                result_row = {
                    "idx":          idx,
                    "title":        title,
                    "model":        model_name,
                    "geval_score":  metric.score,
                    "geval_reason": metric.reason
                }
                print(f"[OK] idx={idx} | model={model_name} | score={metric.score:.3f}")

            except Exception as e:
                print(f"[ERROR] idx={idx} | model={model_name} → {e}")
                result_row = {
                    "idx":          idx,
                    "title":        title,
                    "model":        model_name,
                    "geval_score":  None,
                    "geval_reason": f"error: {str(e)}"
                }

        # Salva incrementalmente — protege contra crashes
        temp_df = pd.DataFrame([result_row])
        if os.path.exists(output_file):
            temp_df.to_csv(output_file, mode="a", header=False, index=False)
        else:
            temp_df.to_csv(output_file, mode="w", header=True, index=False)

print(f"\n✅ Avaliação concluída! Resultados salvos em '{output_file}'")

[OK] idx=0 | model=gemma-3-4b-it | score=0.300


[OK] idx=0 | model=gemma-3-12b-it | score=0.500


[OK] idx=0 | model=gemma-3-27b-it | score=0.600


[OK] idx=0 | model=llama-3.3-70B-Instruct | score=0.400


[OK] idx=0 | model=DeepSeek-R1-Distill-Llama-70B | score=0.200


...

[OK] idx=29 | model=DeepSeek-R1-Distill-Qwen-14B | score=0.400


[OK] idx=29 | model=Phi-4-mini-instruct | score=0.400


[OK] idx=29 | model=qwen/qwen3-4b-2507 | score=0.700


[OK] idx=29 | model=gpt-oss-120b | score=0.800


[OK] idx=29 | model=gpt-oss-20b | score=0.600

✅ Avaliação concluída! Resultados salvos em 'geval_results_gpt.csv'
